# RAG avec LangChain — Questions-Réponses local (sans clé API)

Ce notebook construit un système de **questions-réponses RAG** (Retrieval-Augmented Generation) :
- **Framework :** LangChain
- **Base de connaissances :** dataset public `m-ric/huggingface_doc` (Hugging Face Datasets)
- **Embeddings :** `sentence-transformers/all-MiniLM-L6-v2`
- **Stockage vectoriel :** FAISS
- **LLM local :** `google/flan-t5-small`

Tout s'exécute **localement** après téléchargement — aucune clé API requise. Exécution recommandée sur **Google Colab**.

## 1. Configuration et importations

On épingle **LangChain 0.3.x** : c'est la version où la chaîne `RetrievalQA` utilisée dans cet
exercice est disponible via `from langchain.chains import RetrievalQA`.

💡 Si Colab affiche un avertissement de dépendances après l'installation, faites
*Exécution → Redémarrer la session*, puis relancez à partir de la cellule d'imports.

In [ ]:
%%capture
!pip install -q \
  "langchain==0.3.27" \
  "langchain-community==0.3.27" \
  "langchain-huggingface==0.3.1" \
  "langchain-text-splitters==0.3.11" \
  "faiss-cpu" \
  "sentence-transformers" \
  "transformers" \
  "datasets" 

In [ ]:
# Chargement des données
from datasets import load_dataset

# Composants LangChain
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

# Pipeline Hugging Face
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

print("Imports OK")

## 2. Charger l'ensemble de données

On charge un petit sous-ensemble (`train[:200]`) pour accélérer l'exécution.
Le dataset a deux colonnes : **`text`** (contenu) et **`source`** (provenance, utile pour les citations).

In [ ]:
dataset = load_dataset("m-ric/huggingface_doc", split="train[:200]")

print("Colonnes :", dataset.column_names)
print("Nombre de lignes :", len(dataset))
print("\n--- Exemple de ligne ---")
print("SOURCE:", dataset[0]["source"])
print("TEXT (300 premiers caractères):\n", dataset[0]["text"][:300])

## 3. Convertir les lignes en documents LangChain

Chaque ligne devient un `Document` : le contenu va dans `page_content` et la provenance
dans `metadata["source"]` (pour tracer l'origine des fragments récupérés).

In [ ]:
documents = [
    Document(page_content=row["text"], metadata={"source": row["source"]})
    for row in dataset
]

print("Documents créés :", len(documents))
print("Exemple de metadata :", documents[0].metadata)

## 4. Découper les documents en blocs (chunks)

On segmente avec `RecursiveCharacterTextSplitter`. On teste **deux configurations** pour observer
l'effet du découpage sur le nombre (et la granularité) des chunks.

In [ ]:
# Configuration A : petits chunks
splitter_a = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks_a = splitter_a.split_documents(documents)

# Configuration B : chunks plus grands, davantage de chevauchement
splitter_b = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks_b = splitter_b.split_documents(documents)

print(f"Config A (size=500,  overlap=50 ) -> {len(chunks_a)} chunks")
print(f"Config B (size=1000, overlap=200) -> {len(chunks_b)} chunks")

**Observation :** des `chunk_size` plus petits produisent **plus** de chunks (fragments courts,
récupération plus ciblée mais contexte plus fragmenté) ; des chunks plus grands en produisent
**moins** (plus de contexte par fragment, mais risque d'inclure du texte hors-sujet). Le
`chunk_overlap` évite de couper une idée en deux à la frontière des chunks.

In [ ]:
# On retient la configuration A pour la suite (modifiable pour expérimenter)
chunks = chunks_a
print("Chunks utilisés :", len(chunks))

## 5. Construire le stockage et le récupérateur de vecteurs

On intègre les chunks avec `all-MiniLM-L6-v2`, on crée un **vector store FAISS**, puis un
**retriever**. On teste `k = 2, 4, 6` pour voir l'effet sur la couverture.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Vector store FAISS à partir des chunks
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vector store FAISS construit avec", vectorstore.index.ntotal, "vecteurs")

In [ ]:
# Retriever par défaut : k=4
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Comparaison de différentes valeurs de k
test_question = "What is the Transformers library used for?"
for k in (2, 4, 6):
    r = vectorstore.as_retriever(search_kwargs={"k": k})
    hits = r.invoke(test_question)
    print(f"k={k} -> {len(hits)} chunks récupérés | sources: {[h.metadata['source'].split('/')[-1] for h in hits]}")

Augmenter `k` améliore la **couverture** (plus de contexte fourni au LLM) mais peut introduire
des fragments moins pertinents (**bruit**). Une valeur autour de `k=4` est un bon compromis.

## 6. Vérification de cohérence de la récupération (obligatoire)

Avant de générer des réponses, on **inspecte** les chunks récupérés pour une question donnée.
S'ils ne sont pas pertinents, il faut ajuster le découpage et/ou `k` avant de continuer.

In [ ]:
sanity_question = "How do I load a dataset with the datasets library?"
retrieved = retriever.invoke(sanity_question)

print(f"Question : {sanity_question}\n")
print(f"{len(retrieved)} chunks récupérés :\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} | source: {doc.metadata['source']} ---")
    print(doc.page_content[:300].strip(), "\n")

**À vérifier :** les fragments ci-dessus parlent-ils bien de chargement de datasets ? Si oui,
la récupération est cohérente et on peut passer à la génération. Sinon, revenez à l'étape 4/5
pour ajuster `chunk_size`, `chunk_overlap` ou `k`.

## 7. Réponse aux questions RAG

On configure `google/flan-t5-small` (modèle **seq2seq**, tâche `text2text-generation`), on
construit une chaîne `RetrievalQA`, puis on pose plusieurs questions en affichant la réponse
**et les sources** (pour vérifier que le modèle répond à partir du dataset).

In [ ]:
model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# flan-t5 est un modèle seq2seq -> tâche text2text-generation
gen_pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)
print("LLM local prêt :", model_id)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,   # pour afficher les sources
)
print("Chaîne RetrievalQA construite.")

In [ ]:
def ask(question):
    result = qa_chain.invoke({"query": question})
    print(f"❓ Question : {question}")
    print(f"💬 Réponse : {result['result']}\n")
    print("📚 Sources des fragments utilisés :")
    seen = set()
    for doc in result["source_documents"]:
        src = doc.metadata["source"]
        if src not in seen:
            print("   -", src)
            seen.add(src)
    print("=" * 80, "\n")

questions = [
    "What is the Hugging Face Transformers library used for?",
    "How can I load a dataset using the datasets library?",
    "What is a pipeline in Hugging Face?",
]

for q in questions:
    ask(q)

### Débogage & expérimentation

- Comparez les réponses avec **`chunks_b`** (chunks plus grands) : reconstruisez le vector store
  avec `chunks_b` et relancez.
- Faites varier **`k`** dans le retriever et observez si les réponses gagnent en précision ou en bruit.
- Vérifiez toujours les **sources** : si elles ne correspondent pas à la question, le modèle risque
  de « deviner » plutôt que de s'appuyer sur le dataset.

## ✅ Récapitulatif

Vous avez construit un pipeline RAG complet et 100 % local : chargement du dataset
`m-ric/huggingface_doc`, conversion en `Document` avec métadonnées `source`, découpage en chunks
(en comparant deux stratégies), embeddings `all-MiniLM-L6-v2` + vector store **FAISS**, retriever
paramétrable (`k`), **vérification de cohérence** de la récupération, puis génération de réponses
avec `flan-t5-small` via une chaîne **`RetrievalQA`** affichant les sources.